# TP – Systèmes de recommandation

**Cours** : INF5063 – Machine learning : applications (G. Nollet, L. Benedetti)
**Auteur** : Alban Rouault

Objectif : concevoir et évaluer un système de recommandation de films par filtrage collaboratif
*user-based* sur le jeu de données Kaggle
[The Movies Dataset](https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset).

## Imports et configuration

Toutes les dépendances sont déclarées dans `pyproject.toml` et gérées avec `uv`
(`uv run jupyter lab` pour lancer le notebook).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Affichage des DataFrames
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 60)

# Reproductibilité (séparation train/test, tirages aléatoires)
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

# Emplacement des données
DATA_DIR = Path("Ressources/dataset")

## Exercice 1 – Chargement des données

### 1) Chargement des fichiers

> Récupérer sur Kaggle et charger les fichiers suivants du dataset : `movies_metadata.csv`, `ratings_small.csv` et `links_small.csv`.

In [ ]:
movies = pd.read_csv(DATA_DIR / "movies_metadata.csv", low_memory=False)
ratings = pd.read_csv(DATA_DIR / "ratings_small.csv")
links = pd.read_csv(DATA_DIR / "links_small.csv")

tables = {"movies_metadata": movies, "ratings_small": ratings, "links_small": links}
for name, df in tables.items():
    print(f"{name:<16} {df.shape[0]:>7,} lignes  x {df.shape[1]:>2} colonnes")

**Réponse.** Le dataset [The Movies Dataset](https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset)
a été téléchargé depuis Kaggle et décompressé dans `Ressources/dataset/` (non versionné, voir le README).
Les trois fichiers sont chargés avec pandas. `movies_metadata.csv` est lu avec `low_memory=False` car
quelques lignes mal formées mélangent les types dans certaines colonnes (dont `id`) ; le nettoyage est
fait à la question 5.

### 2) Aperçu des tables

> Afficher un aperçu de chaque table et vérifier leur compréhension.

In [ ]:
for name, df in tables.items():
    print(f"── {name} : {df.shape[0]:,} lignes x {df.shape[1]} colonnes")
    display(df.head(3))

In [ ]:
def resume_colonnes(df: pd.DataFrame) -> pd.DataFrame:
    """Type, taux de remplissage, cardinalité et exemple pour chaque colonne."""
    return pd.DataFrame({
        "type": df.dtypes.astype(str),
        "non nuls": df.notna().sum(),
        "% nuls": (df.isna().mean() * 100).round(1),
        "valeurs distinctes": df.nunique(),
        "exemple": df.iloc[0].astype(str).str.slice(0, 50),
    })

resume_colonnes(movies)

In [ ]:
print("Valeurs possibles de rating :", sorted(ratings["rating"].unique().tolist()))
print("Période des avis :",
      pd.to_datetime(ratings["timestamp"].min(), unit="s").date(), "->",
      pd.to_datetime(ratings["timestamp"].max(), unit="s").date())
print("Doublons (userId, movieId) :", ratings.duplicated(["userId", "movieId"]).sum())
print("Valeurs manquantes dans ratings :", ratings.isna().sum().sum())
print("Valeurs manquantes dans links :", links.isna().sum().to_dict())

**Réponse.**

- **`movies_metadata`** (45 466 lignes, 24 colonnes) : une ligne par film du catalogue TMDB.
  On y trouve l'identifiant TMDB (`id`), l'identifiant IMDB (`imdb_id`), le titre, la date de sortie,
  le budget, les recettes (`revenue`), la durée, la langue, la note moyenne TMDB (`vote_average`) et le
  nombre de votes. Plusieurs colonnes (`genres`, `production_companies`, `belongs_to_collection`…)
  contiennent des listes de dictionnaires stockées sous forme de texte. Certaines colonnes numériques
  (`budget`, `popularity`, `id`) sont lues comme des chaînes à cause de lignes mal formées, ce que
  confirme le tableau de résumé. Des colonnes comme `homepage`, `tagline` ou `belongs_to_collection`
  sont très peu remplies.
- **`ratings_small`** (100 004 lignes, 4 colonnes) : une ligne par avis. `userId` et `movieId` sont
  des identifiants **MovieLens**, `rating` est une note de 0,5 à 5 par pas de 0,5, `timestamp` est une
  date Unix (avis de 1995 à 2016). Aucune valeur manquante, aucun doublon (utilisateur, film) : chaque
  utilisateur n'a noté chaque film qu'une seule fois.
- **`links_small`** (9 125 lignes, 3 colonnes) : une ligne par film du jeu réduit, avec ses trois
  identifiants `movieId` (MovieLens), `imdbId` et `tmdbId`. 13 films n'ont pas de `tmdbId`.

Les avis utilisent donc un espace d'identifiants (MovieLens) différent de celui des métadonnées (TMDB).

### 3) Effectifs

> Combien d'utilisateurs y a-t-il ? De films ? D'avis ?

In [ ]:
n_users = ratings["userId"].nunique()
n_movies_rated = ratings["movieId"].nunique()
n_ratings = len(ratings)
n_movies_links = links["movieId"].nunique()
n_movies_catalog = movies["id"].nunique()

print(f"Utilisateurs                     : {n_users:>7,}")
print(f"Films notés (ratings_small)      : {n_movies_rated:>7,}")
print(f"Films du jeu réduit (links_small): {n_movies_links:>7,}")
print(f"Films du catalogue (metadata)    : {n_movies_catalog:>7,}")
print(f"Avis                             : {n_ratings:>7,}")
print()
print(f"Avis par utilisateur : moyenne {n_ratings / n_users:.0f}, "
      f"médiane {ratings.groupby('userId').size().median():.0f}, "
      f"min {ratings.groupby('userId').size().min()}, "
      f"max {ratings.groupby('userId').size().max()}")
print(f"Densité de la matrice utilisateurs x films : {n_ratings / (n_users * n_movies_rated):.2%}")

**Réponse.** Le jeu réduit contient **671 utilisateurs**, **100 004 avis** et **9 066 films notés**.
Le nombre de films dépend de la table considérée : 9 125 films dans `links_small` (59 n'ont reçu aucun
avis) et 45 466 lignes (45 436 identifiants distincts avant nettoyage) dans `movies_metadata`, qui décrit tout le catalogue TMDB et pas seulement les
films du jeu réduit.

Chaque utilisateur a noté au moins 20 films (médiane 71, un utilisateur en a noté 2 391). La matrice
utilisateurs × films n'est remplie qu'à **1,64 %** : c'est une matrice très creuse, ce qui sera le
principal enjeu du filtrage collaboratif.

### 4) Rôle de `links_small.csv`

> Quelle est l'utilité du fichier `links_small.csv` par rapport aux deux autres fichiers ?

In [ ]:
# Le film movieId = 1 dans ratings : quel est-il ?
tmdb_id = int(links.loc[links["movieId"] == 1, "tmdbId"].iloc[0])
print(f"ratings movieId = 1  ->  links tmdbId = {tmdb_id}  ->  movies_metadata :")
display(movies.loc[movies["id"] == str(tmdb_id), ["id", "imdb_id", "title", "release_date"]])

# Couverture de la table de correspondance
print("movieId de ratings absents de links :", (~ratings["movieId"].isin(links["movieId"])).sum())
print("Films de links sans tmdbId          :", links["tmdbId"].isna().sum())
print("tmdbId de links absents de metadata :",
      (~links["tmdbId"].dropna().astype(int).astype(str).isin(movies["id"])).sum())

**Réponse.** Les deux tables principales n'ont **aucune clé commune** : `ratings_small` identifie
les films par leur `movieId` MovieLens, alors que `movies_metadata` les identifie par leur `id` TMDB
(et leur `imdb_id`). `links_small` est la **table de correspondance** entre ces trois espaces
d'identifiants : c'est elle qui permet de passer d'un avis à un titre de film (et à ses métadonnées),
comme le montre l'exemple ci-dessus (`movieId` 1 → `tmdbId` 862 → *Toy Story*).

Elle sert aussi de **périmètre** : tous les `movieId` notés y figurent, et la version *small* ne
contient que les 9 125 films du jeu réduit, contrairement à `links.csv` qui couvre les 45 000 films.
Ses limites : 13 films n'ont pas de `tmdbId` et 30 `tmdbId` n'ont pas de ligne dans les
métadonnées ; ces films pourront être notés et recommandés, mais pas affichés avec leur titre.